# Assignment 3 — Dimensionality Reduction on MNIST

**Course:** Machine Learning (2026W)  
**Author:** Will Sutherland  
**Dataset:** [MNIST Handwritten Digits](http://yann.lecun.com/exdb/mnist/) — 70,000 grayscale images (28×28 px = 784 features), 10 classes (digits 0–9)

---

## Overview

This notebook investigates how **dimensionality reduction** affects classification performance and interpretability on the MNIST dataset, comparing three techniques: **PCA**, **t-SNE**, and **Locally Linear Embedding (LLE)**.

**Key topics covered:**
- Baseline Random Forest on 784 raw pixel features
- PCA for compression: 784 → 154 dimensions (95% variance retained) and its effect on RF accuracy and training time  
- t-SNE for 2D visualization: cluster separation quality and trustworthiness
- t-SNE vs. LLE: a four-way evaluation (visual quality, trustworthiness, KNN accuracy, runtime)


## 0 · Imports

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, LocallyLinearEmbedding, trustworthiness
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 123


## 1 · Data Preparation

**Train/test split is position-based** (first 60,000 rows = train, next 10,000 = test) following the standard MNIST convention.

**Data note:** Download `mnist_dataset.csv` and update `DATA_PATH` below.


In [ ]:
DATA_PATH = r"mnist_dataset.csv"

df = pd.read_csv(DATA_PATH).iloc[:, 1:]   # drop index column

train = df.iloc[:60_000]
test  = df.iloc[60_000:70_000]

X_train, y_train = train.drop(columns=["label"]), train["label"]
X_test,  y_test  = test.drop(columns=["label"]),  test["label"]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Classes: {sorted(y_train.unique())}")


## 2 · Baseline — Random Forest on All 784 Features

We establish a performance ceiling before applying any dimensionality reduction.

In [ ]:
start = time.time()
rf_full = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_full.fit(X_train, y_train)
t_full = time.time() - start

y_pred_full = rf_full.predict(X_test)
acc_full = accuracy_score(y_test, y_pred_full)

print(f"Training time : {t_full:.2f}s")
print(f"Test accuracy : {acc_full:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_full))


**Result:** 97.01% accuracy in ~9s. The confusion matrix shows the greatest errors on visually similar pairs: **4/9**, **3/5**, and **8/3**.

## 3 · PCA Compression + Random Forest

We retain **95% of explained variance**, reducing the feature space from 784 to ~154 dimensions.  
Key question: does this speed up training, and at what cost to accuracy?


In [ ]:
# ── Fit PCA on training data only (no leakage) ─────────────────────────────
pca = PCA(n_components=0.95, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f"Original features  : {X_train.shape[1]}")
print(f"PCA features       : {X_train_pca.shape[1]}")
print(f"Variance retained  : {pca.explained_variance_ratio_.sum():.4f}")


In [ ]:
start = time.time()
rf_pca = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_pca.fit(X_train_pca, y_train)
t_pca = time.time() - start

y_pred_pca = rf_pca.predict(X_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)

print(f"Training time : {t_pca:.2f}s")
print(f"Test accuracy : {acc_pca:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_pca))


**Result:** Accuracy drops from 97.01% → 95.10%, and training time *increases* (9s → ~18s).

**Why does PCA slow down RF?**  
Random Forests handle high-dimensional sparse data efficiently via random feature subsampling. PCA adds transformation overhead and produces dense, correlated components — removing the sparsity advantage. Linear models and SVMs benefit more from PCA; tree ensembles less so.

The 1.9% accuracy loss reflects information in the discarded 5% of variance that contained discriminative signal.


## 4 · t-SNE Visualization

t-SNE is a non-linear embedding method suited for visualizing high-dimensional data. We run it on a 10,000-sample subset due to its O(n²) complexity.

In [ ]:
SAMPLE_SIZE = 10_000
X_sample = X_train.sample(n=SAMPLE_SIZE, random_state=SEED)
y_sample = y_train.loc[X_sample.index]

print(f"Running t-SNE on {SAMPLE_SIZE:,} samples...")
start = time.time()
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sample)
print(f"Completed in {time.time() - start:.1f}s")


In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1],
                      c=y_sample, cmap="tab10", alpha=0.5, s=5)
plt.colorbar(scatter, label="Digit")
plt.title("t-SNE Projection of MNIST (10,000 samples)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.tight_layout()
plt.show()


**Observations:**
- Most digit clusters are **well-separated** in 2D — t-SNE has successfully preserved 784D structure
- **4/9** and **3/5** show the expected overlap (visually similar digits in handwriting)
- **Digit 1** forms a long, tight cluster — low intra-class variation
- **Digit 0** is compact and cleanly isolated — consistent round shape
- These overlaps mirror the RF confusion matrix, validating both analyses


## 5 · t-SNE vs. Locally Linear Embedding (LLE)

We compare both methods across four criteria:
1. **Visual cluster quality** (scatter plots)
2. **Trustworthiness score** — do 2D neighbours correspond to true 784D neighbours?
3. **KNN accuracy on 2D embeddings** — does the embedding preserve class structure?
4. **Runtime**


In [ ]:
# ── Run both methods on same sample ────────────────────────────────────────
print("Running t-SNE...")
start = time.time()
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sample)
t_tsne = time.time() - start
print(f"  Done in {t_tsne:.1f}s")

print("Running LLE...")
start = time.time()
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=15, random_state=SEED, n_jobs=-1)
X_lle = lle.fit_transform(X_sample)
t_lle = time.time() - start
print(f"  Done in {t_lle:.1f}s")


In [ ]:
# ── Side-by-side scatter plots ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, X_emb, name, t in [
    (axes[0], X_tsne, "t-SNE", t_tsne),
    (axes[1], X_lle,  "LLE",   t_lle),
]:
    sc = ax.scatter(X_emb[:, 0], X_emb[:, 1],
                    c=y_sample, cmap="tab10", alpha=0.5, s=5)
    ax.set_title(f"{name}  ({t:.1f}s)")
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    plt.colorbar(sc, ax=ax, label="Digit")
plt.suptitle("t-SNE vs LLE — MNIST 2D Projection (10,000 samples)", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Trustworthiness scores ─────────────────────────────────────────────────
t_trust = trustworthiness(X_sample, X_tsne, n_neighbors=15)
l_trust = trustworthiness(X_sample, X_lle,  n_neighbors=15)
print(f"t-SNE trustworthiness : {t_trust:.4f}")
print(f"LLE  trustworthiness  : {l_trust:.4f}")


In [ ]:
# ── KNN accuracy on 2D embeddings ──────────────────────────────────────────
print(f"\n{'Method':<8}  {'KNN Accuracy':>13}")
for name, X_emb in [("t-SNE", X_tsne), ("LLE", X_lle)]:
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_emb, y_sample)
    score = knn.score(X_emb, y_sample)
    print(f"{name:<8}  {score:>13.4f}")


## 6 · Summary & Key Takeaways

| Method | Runtime | Trustworthiness | KNN Accuracy |
|---|---|---|---|
| **t-SNE** | ~100s | **0.98** | **0.957** |
| LLE | ~65s | 0.82 | 0.728 |

**t-SNE is the clear winner on embedding quality.** Its near-perfect trustworthiness score (0.98) means 2D neighbours almost always correspond to true 784D neighbours. The KNN accuracy on t-SNE coordinates (95.7%) approaches the full RF baseline (97%), confirming that t-SNE compresses the data while preserving nearly all class structure.

**LLE is ~35% faster** but its clusters overlap substantially, and its KNN accuracy (72.8%) signals significant structural distortion.

**PCA vs. t-SNE:** PCA is a linear method preserving global variance; t-SNE is non-linear and preserves local neighbourhood structure. For visualization and cluster discovery, t-SNE is far superior. For preprocessing before a downstream classifier, PCA's interpretability and invertibility make it more practical — though for Random Forests, the gains are modest or negative.
